In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv("SMSSpamCollection",sep='\t',names=['label','message'])

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   int64 
 1   message  5572 non-null   object
dtypes: int64(1), object(1)
memory usage: 87.2+ KB


In [ ]:
#Rule for NN when it comes to text data
#1. ANN/CNN understands only padded sequence(NUMBERS) (You need to represent text data into numbers)
#2. Input Size(Number of Tokens) must be FIXED
#3. Labels must be discrete numerical/binarized

In [ ]:
#Binarize labels
data['label'] = data['label'].map({'ham':0, 'spam': 1})

In [ ]:
data.head()

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
#Seperate data as features and label
features = data.message.values
label = data.label.values

In [ ]:
label.shape

(5572,)

In [ ]:
#Train test split

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(features,
                                                 label,
                                                 test_size=0.2,
                                                 random_state=1)

In [ ]:
#Start with Tokenization
from tensorflow.keras.preprocessing.text import Tokenizer

#Decide the Vocabulary Word Frequency Size. Used only when Regularization is required

vocabFreqWordSize = None

#Convert the sentences into sequence of words

tokenizer = Tokenizer(num_words=vocabFreqWordSize, oov_token="<DontKnow>", split=" ")

#Fit tokenizer with training set

tokenizer.fit_on_texts(X_train)

In [ ]:
tokenizer.word_index

{'<DontKnow>': 1,
 'i': 2,
 'to': 3,
 'you': 4,
 'a': 5,
 'the': 6,
 'u': 7,
 'and': 8,
 'is': 9,
 'in': 10,
 'me': 11,
 'my': 12,
 'for': 13,
 'your': 14,
 'it': 15,
 'of': 16,
 'call': 17,
 'have': 18,
 'that': 19,
 'on': 20,
 '2': 21,
 'now': 22,
 'are': 23,
 'so': 24,
 'not': 25,
 'but': 26,
 'can': 27,
 'do': 28,
 'or': 29,
 'if': 30,
 'get': 31,
 "i'm": 32,
 'ur': 33,
 'with': 34,
 'at': 35,
 'be': 36,
 'will': 37,
 'just': 38,
 'no': 39,
 'this': 40,
 'we': 41,
 '4': 42,
 'gt': 43,
 'lt': 44,
 'up': 45,
 'when': 46,
 'from': 47,
 'go': 48,
 'ok': 49,
 'free': 50,
 'how': 51,
 'out': 52,
 'all': 53,
 'what': 54,
 'know': 55,
 'like': 56,
 'good': 57,
 'got': 58,
 'then': 59,
 'its': 60,
 'only': 61,
 'was': 62,
 'am': 63,
 'come': 64,
 'time': 65,
 'day': 66,
 'love': 67,
 'want': 68,
 'text': 69,
 'send': 70,
 'he': 71,
 'as': 72,
 'by': 73,
 'there': 74,
 'going': 75,
 'about': 76,
 'one': 77,
 'ü': 78,
 'txt': 79,
 "i'll": 80,
 'stop': 81,
 'need': 82,
 'home': 83,
 'r': 84,
 

In [ ]:
#Lets create Sequence object

seqTrain = tokenizer.texts_to_sequences(X_train)

In [ ]:
seqTest = tokenizer.texts_to_sequences(X_test)

In [ ]:
#Lets pad the sequence data

from tensorflow.keras.preprocessing.sequence import pad_sequences
train_data = pad_sequences(seqTrain)
T = train_data.shape[1]
T

189

In [ ]:
testData = pad_sequences(seqTest,maxlen=T)

In [ ]:
train_data.shape

(4457, 189)

In [ ]:
testData.shape

(1115, 189)

In [ ]:
len(tokenizer.word_index)

7969

In [ ]:
#Modelling Phase

import tensorflow as tf

vocabSize=len(tokenizer.word_index)
maxlen=T
embeddingDimension = 20 #Hyperparameter any value between 10 to inf (Natural no)

In [ ]:
model = tf.keras.Sequential()
#Embedding converts sequence data into a dense vector. Embedding is responsible to preserve the semantic meaning of the sentence

model.add(tf.keras.layers.Embedding(vocabSize + 1, embeddingDimension, input_length=maxlen))

model.add(tf.keras.layers.Conv1D(filters=128, kernel_size=5, activation="relu"))

model.add(tf.keras.layers.GlobalAveragePooling1D())

model.add(tf.keras.layers.Dense(24, activation="relu"))
model.add(tf.keras.layers.Dense(12, activation="relu"))
model.add(tf.keras.layers.Dense(1, activation="sigmoid"))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
#Custom Callback for early stopping

class MyThresholdCallBack(tf.keras.callbacks.Callback):
    def __init__(self,cl):
        super(MyThresholdCallBack, self).__init__()
        self.cl = cl

    def on_epoch_end(self, epoch, logs=None):
        test_score = logs["val_accuracy"]
        train_score = logs["accuracy"]

        if test_score > train_score and test_score > self.cl:
        #if test_score > self.cl:
            self.model.stop_training = True

In [ ]:
myAccuracyMonitor = MyThresholdCallBack(cl=0.9)

In [ ]:
model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=['accuracy',tf.keras.metrics.F1Score(num_classes=1, threshold=0.5, average='micro')])

In [ ]:
model.fit(train_data,y_train, epochs=10, validation_data=(testData,y_test), callbacks=[myAccuracyMonitor])

Epoch 1/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8634 - loss: 0.4279 - val_accuracy: 0.8682 - val_loss: 0.3737
Epoch 2/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8581 - loss: 0.3919 - val_accuracy: 0.8682 - val_loss: 0.3665
Epoch 3/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8726 - loss: 0.3563 - val_accuracy: 0.8682 - val_loss: 0.3490
Epoch 4/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8644 - loss: 0.3480 - val_accuracy: 0.8682 - val_loss: 0.2997
Epoch 5/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8573 - loss: 0.3099 - val_accuracy: 0.9157 - val_loss: 0.2160
Epoch 6/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9169 - loss: 0.1962 - val_accuracy: 0.9686 - val_loss: 0.1344
Epoch 7/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9674 - loss: 0.1297 - val_accuracy: 0.9704 - val_loss: 0.0817
Epoch 8/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9754 - loss: 0.0847 - val_accuracy: 0

In [ ]:
#Assignment 1
#==============


#Using yelp dataset, create a model that can understand the sentiment of the user.

#Feature: text
#Label: stars


#Interpretation of stars
# 5,4,3 -> Happy
# 2,1 -> Sad

# ML,ANN, CNN